# Assignment No. - 1
**Name:** Kadam Abhijeet Jalindar &nbsp;&nbsp;&nbsp;&nbsp; **Roll No.:** COA426 &nbsp;&nbsp;&nbsp;&nbsp; **Div:** A

**Title of the Assignment:** Linear regression by using Deep Neural network: Implement Boston housing price prediction problem by Linear regression using Deep Neural network. Use Boston House price prediction dataset.

In [ ]:
# Importing the pandas for data processing and numpy for numerical computing
import numpy as np
import pandas as pd

In [ ]:
!pip install scikit-learn==1.1.3

In [ ]:
# Importing the Boston Housing dataset from the sklearn
from sklearn.datasets import load_boston
boston = load_boston()

In [ ]:
# Converting the data into pandas dataframe
data = pd.DataFrame(boston.data)

# First look at the data
data.head()

In [ ]:
# Adding the feature names to the dataframe
data.columns = boston.feature_names

# Adding the target variable to the dataset
data['PRICE'] = boston.target

# Looking at the data with names and target variable
data.head(n=10)

In [ ]:
# Shape of the data
print(data.shape)

In [ ]:
# Checking the null values in the dataset
data.isnull().sum()

In [ ]:
# Checking the statistics of the data
# This is sometimes very useful, for example if you look at the CRIM the max is
# 88.97 and 75% of the value is below 3.677083 and
# mean is 3.613524 so it means the max values is actually an outlier or there are
# outliers present in the column
data.describe()

In [ ]:
data.info()

In [ ]:
# Checking the distribution of the target variable
import seaborn as sns

sns.distplot(data.PRICE)
# The distribution seems normal. Had the data not been normal we would have performed
# log transformation or took the square root of the data to make the data normal.
# Normal distribution is needed for machine learning for better predictability of the model.

In [ ]:
# Distribution using box plot
sns.boxplot(data.PRICE)

In [ ]:
# Checking the correlation of the independent feature with the dependent feature
# Correlation is a statistical technique that can show whether and how strongly
# pairs of variables are related. An intelligent correlation analysis can lead to
# a greater understanding of your data.

# Checking Correlation of the data
correlation = data.corr()
correlation.loc['PRICE']

In [ ]:
# Plotting the heatmap
import matplotlib.pyplot as plt

fig, axes = plt.subplots(figsize=(15, 12))
sns.heatmap(correlation, square=True, annot=True)
# By looking at the correlation plot LSTAT is negatively correlated with -0.75 and
# RM is positively correlated to the price and PTRATIO is correlated negatively with -0.51

In [ ]:
# Checking the scatter plot with the most correlated features
plt.figure(figsize=(20, 5))
features = ['LSTAT', 'RM', 'PTRATIO']

for i, col in enumerate(features):
    plt.subplot(1, len(features), i+1)
    x = data[col]
    y = data.PRICE
    plt.scatter(x, y, marker='o')
    plt.title("Variation in House prices")
    plt.xlabel(col)
    plt.ylabel('"House prices in $1000"')

In [ ]:
# Splitting the dependent feature and independent feature
# X = data[['LSTAT','RM','PTRATIO']]
X = data.iloc[:, :-1]
y = data.PRICE

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# In order to provide a standardized input to our neural network, we need to
# perform the normalization of our dataset.
# This can be seen as a step to reduce the differences in scale that may arise
# from the existent features.
# We perform this normalization by subtracting the mean from our data and
# dividing it by the standard deviation.
# One more time, this normalization should only be performed by using the mean
# and standard deviation from the training set,
# in order to avoid any information leak from the test set.

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [ ]:
# Linear Regression
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()

# Fitting the model
regressor.fit(X_train, y_train)

In [ ]:
# Model Evaluation
# Prediction on the test dataset
y_pred = regressor.predict(X_test)

# Predicting RMSE on the Test set results
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)
print(r2)

In [ ]:
# Neural Networks
# Scaling the dataset
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
# Due to the small amount of presented data in this dataset, we must be careful
# to not create an overly complex model, which could lead to overfitting our data.
# For this, we are going to adopt an architecture based on two Dense layers,
# the first with 128 and the second with 64 neurons, both using a ReLU activation function.
# A dense layer with a linear activation will be used as output layer.
# In order to allow us to know if our model is properly learning, we will use a
# mean squared error loss function and to report the performance we will adopt the
# mean average error metric.

# Creating the neural network model
import keras
from keras.layers import Dense, Activation, Dropout
from keras.models import Sequential

model = Sequential()
model.add(Dense(128, activation='relu', input_dim=13))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

In [ ]:
!pip install ann_visualizer
!pip install graphviz

from ann_visualizer.visualize import ann_viz
ann_viz(model, title="DEMO ANN")

In [ ]:
history = model.fit(X_train, y_train, epochs=100, validation_split=0.05)

In [ ]:
# By plotting both loss and mean average error, we can see that our model was capable of
# learning patterns in our data without overfitting taking place
# (as shown by the validation set curves)

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scattergl(y=history.history['loss'], name='Train'))
fig.add_trace(go.Scattergl(y=history.history['val_loss'], name='Valid'))
fig.update_layout(height=500, width=700,
                  xaxis_title='Epoch',
                  yaxis_title='Loss')
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scattergl(y=history.history['mae'], name='Train'))
fig.add_trace(go.Scattergl(y=history.history['val_mae'], name='Valid'))
fig.update_layout(height=500, width=700,
                  xaxis_title='Epoch',
                  yaxis_title='Mean Absolute Error')
fig.show()

In [ ]:
# Evaluation of the model
y_pred = model.predict(X_test)

mse_nn, mae_nn = model.evaluate(X_test, y_test)
print('Mean squared error on test data: ', mse_nn)
print('Mean absolute error on test data: ', mae_nn)

In [ ]:
# Comparison with traditional approaches
# First let's try with a simple algorithm, the Linear Regression:
from sklearn.metrics import mean_absolute_error

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print('Mean squared error on test data: ', mse_lr)
print('Mean absolute error on test data: ', mae_lr)

r2 = r2_score(y_test, y_pred)
print(r2)

In [ ]:
# Predicting RMSE on the Test set results
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

In [ ]:
# Make predictions on new data
import sklearn

new_data = sklearn.preprocessing.StandardScaler().fit_transform(
    [[0.1, 10.0, 5.0, 0, 0.4, 6.0, 50, 6.0, 1, 400, 20, 300, 10]]
)

prediction = model.predict(new_data)
print("Predicted house price:", prediction)

# new_data = sklearn.preprocessing.StandardScaler().fit_transform(...) is a line of code
# that standardizes the input features of a new data point.
# In this specific case, we have a new data point represented as a list of 13 numeric values
# ([0.1, 10.0, 5.0, 0, 0.4, 6.0, 50, 6.0, 1, 400, 20, 300, 10]) that represents the values
# for the 13 features of the Boston House Price dataset.
# The StandardScaler() function from the sklearn.preprocessing module is used to standardize
# the data. Standardization scales each feature to have zero mean and unit variance,
# which is a common preprocessing step in machine learning to ensure that all features
# contribute equally to the model.
# The fit_transform() method is used to fit the scaler to the data and apply the
# standardization transformation. The result is a new data point with standardized feature values.